# Unified SMC Binarized MNIST

Restored code-only notebook for sweeping MDM/ReMDM/UDM, base/grad proposal modes, and multiple guidance-step counts. Allstep warmup and allstep policy are written once per `(MODEL_TYPE, PROPOSAL_MODE)` pair, while reduced policies are written per guidance-step count.


In [ ]:
import math
import random
import re
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
from torch import Tensor
from torchvision import datasets, transforms, utils
from tqdm import tqdm

project_root = Path.cwd().resolve()
if project_root.name == "scripts":
    project_root = project_root.parent
elif not (project_root / "smc").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "smc").exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

%load_ext autoreload
%autoreload 2

from datasets.binarized_mnist import build_dataloaders
from models.denoising_models.unet_with_attention import UNetWithAttention
from models.discrete_diffusion.mdm import MaskedDiffusion
from models.discrete_diffusion.remdm import ReMaskingDiffusion
from models.discrete_diffusion.udm import UniformDiffusion
from models.discrete_diffusion.utils.parametrizations import subs_parametrization_continuous
from models.reward_models.binarized_mnist_classifier import BinarizedMNISTClassifier
from smc.proposals import first_order_approximation_optimal_proposal, reverse_as_proposal
from smc.sampling_algorithms import multinomial_resample, stratified_resample, systematic_resample
from smc.smc import sequential_monte_carlo_base_tstep, sequential_monte_carlo_grad_tstep
from smc.utils import lambda_schedule
from utils.binarized_mnist_utils import (
    binarized_images_diversity,
    binarized_images_unique_representative_indices,
    binarized_images_uniqueness_score,
)

try:
    from utils.plot_utils import VALID_REWARD_THRESHOLD, plot_smc_results_binarized_mnist
except ModuleNotFoundError:
    VALID_REWARD_THRESHOLD = 0.1

    def plot_smc_results_binarized_mnist(result, num_timesteps, vocab_size, num_categories, compute_rewards_fn, *args, **kwargs):
        samples_final = result["X_0"].detach().cpu().long()
        one_hot = F.one_hot(samples_final, num_classes=num_categories).float()
        with torch.no_grad():
            rewards = compute_rewards_fn(one_hot).detach().cpu()
        exp_rewards = torch.exp(rewards)
        mean_exp_rewards = np.asarray(result.get("mean_exp_rewards_trace", []), dtype=float)

        unique_indices = binarized_images_unique_representative_indices(samples_final, 0.85)
        valid_samples = samples_final[rewards > -VALID_REWARD_THRESHOLD]
        return {
            "mean_exp_reward": float(exp_rewards.mean().item()),
            "mean_exp_rewards": mean_exp_rewards,
            "unique_sample_exp_reward_sum": float(exp_rewards[unique_indices].sum().item()) if unique_indices else 0.0,
            "diversity": float(binarized_images_diversity(samples_final.numpy())),
            "uniqueness": int(len(unique_indices)),
            "unique_valid_count": int(binarized_images_uniqueness_score(valid_samples, 0.85)),
        }


## Configuration


In [ ]:
MODEL_TYPE = ["mdm"]       # "mdm", "remdm", or "udm"
PROPOSAL_MODE = ["base", "grad"]            # "base" or "grad"
N_GUIDANCE_STEPS = [3, 6, 9, 12, 15, 18, 21, 24, 27]
REWARD_THRESHOLD = -0.1
SIMILARITY_THRESHOLD = 0.85

CUDA_DEVICE_INDEX = 3
SEED = 42
WARMUP_SEEDS = range(100, 103)
EVAL_SEEDS = range(0, 50)

batch_size = 64
vocab_size = 2
input_shape = (1, 28, 28)
num_timesteps = 100
kl_weight = 1.0

num_particles = 20
ESS_min = 19
reward_estimate_sample_count = 1
use_partial_resampling = True
perform_final_resample = True
partial_resample_size = num_particles // 2

RESULTS_DIR = project_root / "scripts"
RESULTS_DIR.mkdir(exist_ok=True)
RESULT_PREFIX = None

SMC_TSTEP_FNS = {
    "base": sequential_monte_carlo_base_tstep,
    "grad": sequential_monte_carlo_grad_tstep,
}
SMC_TSTEP_FN = None
pretrained_model = None
num_categories = None
mask_index = None


def _sweep_values(value):
    if isinstance(value, (list, tuple, set, range)):
        return tuple(value)
    return (value,)


MODEL_TYPE_SWEEP = _sweep_values(MODEL_TYPE)
PROPOSAL_MODE_SWEEP = _sweep_values(PROPOSAL_MODE)
N_GUIDANCE_STEPS_SWEEP = tuple(int(value) for value in _sweep_values(N_GUIDANCE_STEPS))

if torch.cuda.is_available():
    if CUDA_DEVICE_INDEX is None:
        device = torch.device("cuda")
    else:
        cuda_index = min(CUDA_DEVICE_INDEX, torch.cuda.device_count() - 1)
        device = torch.device(f"cuda:{cuda_index}")
        torch.cuda.set_device(device)
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(
    "Planned sweep configs: "
    f"{len(MODEL_TYPE_SWEEP)} model types x "
    f"{len(PROPOSAL_MODE_SWEEP)} proposal modes x "
    f"{len(N_GUIDANCE_STEPS_SWEEP)} guidance counts = "
    f"{len(MODEL_TYPE_SWEEP) * len(PROPOSAL_MODE_SWEEP) * len(N_GUIDANCE_STEPS_SWEEP)} runs"
)


## Model And Reward Setup


In [ ]:
def build_pretrained_model(model_type: str, device: torch.device):
    model_type = model_type.lower()

    if model_type in {"mdm", "remdm"}:
        num_categories_local = vocab_size + 1
        mask_index_local = num_categories_local - 1
        denoising_model = UNetWithAttention(
            num_categories=num_categories_local,
            embedding_dim=64,
            ch_mult=(2, 4, 8),
            num_res_blocks=2,
            attention_resolutions=(1, 2),
            encode_time=False,
            probs_parametrization_fn=subs_parametrization_continuous,
        )

        if model_type == "mdm":
            model = MaskedDiffusion(
                denoising_model=denoising_model,
                num_categories=num_categories_local,
                input_shape=input_shape,
                mask_index=mask_index_local,
                masking_schedule="linear",
                num_timesteps=num_timesteps,
                discretization_schedule="linear",
            ).to(device)
        else:
            model = ReMaskingDiffusion(
                denoising_model=denoising_model,
                num_categories=num_categories_local,
                input_shape=input_shape,
                mask_index=mask_index_local,
                masking_schedule="linear",
                num_timesteps=num_timesteps,
                discretization_schedule="linear",
                remasking_schedule="max_capped",
                remasking_kwargs={"eta_cap": 0.1, "eta_rescale": 0.1},
            ).to(device)

        weight_path = project_root / "model_weights" / "mdm_binarized_mnist_256.pth"

    elif model_type == "udm":
        num_categories_local = vocab_size
        mask_index_local = None
        model = UniformDiffusion(
            denoising_model=UNetWithAttention(
                num_categories=num_categories_local,
                embedding_dim=64,
                ch_mult=(2, 4, 8),
                num_res_blocks=2,
                attention_resolutions=(1, 2),
                encode_time=True,
            ),
            num_categories=num_categories_local,
            input_shape=input_shape,
            noise_schedule="linear",
            num_timesteps=num_timesteps,
            discretization_schedule="linear",
        ).to(device)
        weight_path = project_root / "model_weights" / "udm_binarized_mnist_256.pth"

    else:
        raise ValueError(f"Unknown MODEL_TYPE: {model_type}")

    model.load_state_dict(torch.load(weight_path, map_location=device))
    model.eval()
    return model, num_categories_local, mask_index_local


def clear_active_model():
    global pretrained_model, num_categories, mask_index
    old_model = globals().get("pretrained_model")
    pretrained_model = None
    num_categories = None
    mask_index = None
    if old_model is not None:
        del old_model
    if device.type == "cuda":
        torch.cuda.empty_cache()


def set_guidance_steps(n_guidance_steps: int):
    global N_GUIDANCE_STEPS
    N_GUIDANCE_STEPS = int(n_guidance_steps)
    return N_GUIDANCE_STEPS


def set_experiment_context(model_type: str, proposal_mode: str, n_guidance_steps: int):
    global MODEL_TYPE, PROPOSAL_MODE, RESULT_PREFIX, SMC_TSTEP_FN
    global pretrained_model, num_categories, mask_index

    clear_active_model()
    MODEL_TYPE = str(model_type).lower()
    PROPOSAL_MODE = str(proposal_mode).lower()
    set_guidance_steps(n_guidance_steps)

    if MODEL_TYPE not in {"mdm", "remdm", "udm"}:
        raise ValueError(f"Unknown MODEL_TYPE: {MODEL_TYPE}")
    if PROPOSAL_MODE not in SMC_TSTEP_FNS:
        raise ValueError(f"Unknown PROPOSAL_MODE: {PROPOSAL_MODE}")

    RESULT_PREFIX = f"{MODEL_TYPE}_{PROPOSAL_MODE}"
    SMC_TSTEP_FN = SMC_TSTEP_FNS[PROPOSAL_MODE]
    pretrained_model, num_categories, mask_index = build_pretrained_model(MODEL_TYPE, device)
    print(
        f"Loaded {MODEL_TYPE} / {PROPOSAL_MODE} / "
        f"N_GUIDANCE_STEPS={N_GUIDANCE_STEPS} with {num_categories=}, {mask_index=}"
    )
    return {
        "model_type": MODEL_TYPE,
        "proposal_mode": PROPOSAL_MODE,
        "n_guidance_steps": N_GUIDANCE_STEPS,
        "num_categories": num_categories,
        "mask_index": mask_index,
    }


mnist_classfier_model = BinarizedMNISTClassifier().to(device)
mnist_classfier_model.load_state_dict(
    torch.load(project_root / "model_weights" / "binarized_mnist_classifier_1.pth", map_location=device)
)
mnist_classfier_model.eval()


def compute_rewards_for_batch(x: Tensor, with_grad: bool = False):
    logits = mnist_classfier_model(x[..., :vocab_size].to(device))
    logits = logits.log_softmax(dim=-1)
    reward = logits[:, 4]
    return reward


def compute_rewards(x: Tensor, with_grad: bool = False):
    n_samples = x.shape[0]
    rewards_all = []
    for i in range(0, n_samples, batch_size):
        if with_grad:
            rewards = compute_rewards_for_batch(x[i:i + batch_size], with_grad=True)
        else:
            with torch.no_grad():
                rewards = compute_rewards_for_batch(x[i:i + batch_size], with_grad=False)
        rewards_all.append(rewards)
    return torch.cat(rewards_all)


def intialize_particles(num_particles, device=device):
    if MODEL_TYPE == "udm":
        return torch.randint(
            low=0,
            high=vocab_size,
            size=(num_particles, *input_shape),
            device=device,
            requires_grad=False,
        )
    return torch.full(
        (num_particles, *input_shape),
        mask_index,
        device=device,
        requires_grad=False,
    )


# lambdas = lambda_schedule(num_timesteps, gamma=0.3)
# lambdas = lambda_schedule(num_timesteps, base=1.0)
lambdas = np.linspace(1.0, 1.0, num_timesteps + 1)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = False
    cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed(SEED)


In [ ]:
# Optional reward preview. This cell is safe before a diffusion model is active.
train_loader, test_loader = build_dataloaders(batch_size=20)
samples, _ = next(iter(train_loader))
rewards = compute_rewards(F.one_hot(samples.long(), num_classes=vocab_size).float())

fig, axes = plt.subplots(1, 20, figsize=(15, 2))
for i in range(20):
    ax = axes[i]
    ax.imshow(samples[i].squeeze().cpu().numpy(), cmap="gray")
    ax.set_title(f"{rewards[i].item():.4f}", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Shared Experiment Helpers


In [ ]:
def result_suffix(n_guidance_steps: int | None = None, shared: bool = False):
    return "shared" if shared else f"T{N_GUIDANCE_STEPS if n_guidance_steps is None else int(n_guidance_steps)}"


def result_path(kind: str, n_guidance_steps: int | None = None, shared: bool = False):
    if RESULT_PREFIX is None:
        raise RuntimeError("Call set_experiment_context(...) before creating result paths.")
    if kind == "allstep":
        return RESULTS_DIR / f"experiment_results_{RESULT_PREFIX}_{kind}.txt"
    suffix = result_suffix(n_guidance_steps=n_guidance_steps, shared=shared)
    return RESULTS_DIR / f"experiment_results_{RESULT_PREFIX}_{kind}_{suffix}.txt"


METRIC_SEMANTICS_VERSION = "exp_reward_v6_fixed_validity_threshold"
METRIC_SEMANTICS_LINE = f"Metric semantics: {METRIC_SEMANTICS_VERSION}"
LEGACY_CACHE_SEMANTICS_LINES = ("Metric semantics: exp_reward_v4_weighted_warmup",)


def required_tstep_policy_names():
    return tuple(policy_name for policy_name, _, _ in globals().get("LATEX_POLICY_ROWS", []) if policy_name != "allstep")


def has_compatible_metric_semantics(path: Path):
    text = path.read_text(encoding="utf-8", errors="replace")
    return METRIC_SEMANTICS_LINE in text or any(line in text for line in LEGACY_CACHE_SEMANTICS_LINES)


def is_metric_cache_compatible(path: Path, kind: str | None = None):
    text = path.read_text(encoding="utf-8", errors="replace")
    if not has_compatible_metric_semantics(path):
        return False
    if kind == "tstep":
        missing = [policy_name for policy_name in required_tstep_policy_names() if f"Summary averages | Policy: {policy_name}" not in text]
        if missing:
            print(f"Ignoring incomplete tstep cache missing policies {missing}: {path}")
            return False
    return True


def find_existing_result_path(kind: str, n_guidance_steps: int | None = None, shared: bool = False):
    output_file = result_path(kind, n_guidance_steps=n_guidance_steps, shared=shared)
    if output_file.exists():
        if is_metric_cache_compatible(output_file, kind=kind):
            return output_file
        print(f"Ignoring incompatible result cache: {output_file}")
    if kind == "allstep":
        return None
    suffix = result_suffix(n_guidance_steps=n_guidance_steps, shared=shared)
    legacy_pattern = f"experiment_results_{RESULT_PREFIX}_{kind}_{suffix}_*.txt"
    legacy_matches = sorted(RESULTS_DIR.glob(legacy_pattern), reverse=True)
    for legacy_path in legacy_matches:
        if is_metric_cache_compatible(legacy_path, kind=kind):
            return legacy_path
        print(f"Ignoring incompatible result cache: {legacy_path}")
    return None


TIMESTEP_MEAN_RE = re.compile(r"Timestep (\d+): mean=([-+0-9.eE]+) std=([-+0-9.eE]+)")
RECORD_HEADER_RE = re.compile(r"Seed: (?P<seed>\d+) \| Policy: (?P<policy>\w+) \| tstep_count: (?P<tstep_count>\d+)")
ELAPSED_LINE_RE = re.compile(r"Total Execution Time: (?P<elapsed>[-+0-9.eE]+) seconds")
METRIC_LINE_RE = re.compile(r"Mean Exp Reward: (?P<mean_exp_reward>[-+0-9.eE]+), Unique Sample Exp Reward Sum: (?P<unique_sample_exp_reward_sum>[-+0-9.eE]+), Diversity: (?P<diversity>[-+0-9.eE]+), Uniqueness \(similarity > 0\.85\): (?P<uniqueness>[-+0-9.eE]+), Unique Valid Count \(r\(x_0\) > -0\.1, similarity > 0\.85\): (?P<unique_valid_count>[-+0-9.eE]+)")


def load_warmup_mean_exp_rewards(output_file: Path):
    values = {}
    text = output_file.read_text(encoding="utf-8", errors="replace")
    for match in TIMESTEP_MEAN_RE.finditer(text):
        timestep = int(match.group(1))
        values[timestep] = (float(match.group(2)), float(match.group(3)))
    if not values:
        raise ValueError(f"No warmup timestep means found in {output_file}")
    max_timestep = max(values)
    means = [values[timestep][0] for timestep in range(max_timestep + 1)]
    stds = [values[timestep][1] for timestep in range(max_timestep + 1)]
    return np.asarray(means), np.asarray(stds)


def parse_result_records(output_file: Path):
    lines = output_file.read_text(encoding="utf-8", errors="replace").splitlines()
    records = []
    index = 0
    while index < len(lines):
        header_match = RECORD_HEADER_RE.fullmatch(lines[index])
        if header_match is None:
            index += 1
            continue
        elapsed_match = ELAPSED_LINE_RE.fullmatch(lines[index + 1]) if index + 1 < len(lines) else None
        metric_match = METRIC_LINE_RE.fullmatch(lines[index + 2]) if index + 2 < len(lines) else None
        if elapsed_match is None or metric_match is None:
            index += 1
            continue
        metrics = {
            "mean_exp_reward": float(metric_match.group("mean_exp_reward")),
            "unique_sample_exp_reward_sum": float(metric_match.group("unique_sample_exp_reward_sum")),
            "diversity": float(metric_match.group("diversity")),
            "uniqueness": float(metric_match.group("uniqueness")),
            "unique_valid_count": int(float(metric_match.group("unique_valid_count"))),
        }
        records.append({
            "seed": int(header_match.group("seed")),
            "policy": header_match.group("policy"),
            "tstep_count": int(header_match.group("tstep_count")),
            "metrics": metrics,
            "elapsed": float(elapsed_match.group("elapsed")),
        })
        index += 3
    return records


def sync_if_cuda():
    if device.type == "cuda":
        torch.cuda.synchronize()


def run_smc_once(seed: int, tstep: set[int]):
    if pretrained_model is None or SMC_TSTEP_FN is None or num_categories is None:
        raise RuntimeError("Call set_experiment_context(...) before running SMC.")

    set_seed(seed)
    sync_if_cuda()
    start_time = time.time()

    results = SMC_TSTEP_FN(
        model=pretrained_model,
        num_categories=num_categories,
        T=num_timesteps,
        N=num_particles,
        ESS_min=ESS_min,
        intialize_particles_fn=intialize_particles,
        resample_fn=systematic_resample,
        use_partial_resampling=use_partial_resampling,
        partial_resample_size=partial_resample_size,
        proposal_fn=first_order_approximation_optimal_proposal,
        compute_reward_fn=compute_rewards,
        lambdas=lambdas,
        kl_weight=kl_weight,
        reward_estimate_sample_count=reward_estimate_sample_count,
        perform_final_resample=perform_final_resample,
        device=device,
        tstep=tstep,
    )

    sync_if_cuda()
    elapsed = time.time() - start_time
    metrics = plot_smc_results_binarized_mnist(results, num_timesteps, vocab_size, num_categories, compute_rewards)
    return results, metrics, elapsed


def get_unique_sample_exp_reward_sum(metrics: dict[str, Any]):
    value = metrics.get("unique_sample_exp_reward_sum")
    if value is not None:
        return value
    old_mean_value = metrics.get("unique_sample_mean_exp_reward")
    uniqueness = metrics.get("uniqueness")
    if old_mean_value is not None and uniqueness is not None:
        return old_mean_value * uniqueness
    return 0.0


def write_metrics(f, metrics: dict[str, Any]):
    unique_sample_exp_reward_sum = get_unique_sample_exp_reward_sum(metrics)
    metrics_info = (
        f"Mean Exp Reward: {metrics['mean_exp_reward']:.4f}, "
        f"Unique Sample Exp Reward Sum: {unique_sample_exp_reward_sum:.4f}, "
        f"Diversity: {metrics['diversity']:.4f}, "
        f"Uniqueness (similarity > 0.85): {metrics['uniqueness']:.4f}, "
        f"Unique Valid Count (r(x_0) > -0.1, similarity > 0.85): {metrics['unique_valid_count']}\n"
    )
    print(metrics_info, end="")
    f.write(metrics_info)


def summarize_mean_exp_rewards(all_mean_exp_rewards):
    if len(all_mean_exp_rewards) == 0:
        return None, None
    min_length = min(len(arr) for arr in all_mean_exp_rewards)
    aligned = [arr[:min_length] for arr in all_mean_exp_rewards]
    stacked = np.stack(aligned, axis=0)
    mean_exp_rewards_avg = np.mean(stacked, axis=0)
    if stacked.shape[0] > 1:
        mean_exp_rewards_std = np.std(stacked, axis=0, ddof=1)
    else:
        mean_exp_rewards_std = np.zeros(stacked.shape[1], dtype=float)
    return np.asarray(mean_exp_rewards_avg)[::-1], np.asarray(mean_exp_rewards_std)[::-1]


def _summary_stats(values):
    values = [value for value in values if value is not None]
    if not values:
        return None
    arr = np.asarray(values, dtype=float)
    std = float(np.std(arr, ddof=1)) if arr.size > 1 else 0.0
    return {"mean": float(np.mean(arr)), "std": std, "min": float(np.min(arr)), "max": float(np.max(arr)), "n": int(arr.size)}


def _write_summary_line(f, line: str = ""):
    print(line)
    f.write(line + "\n")


def _format_summary_stats(name: str, stats: dict[str, Any]):
    return f"{name}: mean={stats['mean']:.4f}, std={stats['std']:.4f}, min={stats['min']:.4f}, max={stats['max']:.4f}, n={stats['n']}"


def write_metric_summary(f, title: str, records: list[dict[str, Any]]):
    if not records:
        return
    _write_summary_line(f, "")
    _write_summary_line(f, "=" * 80)
    _write_summary_line(f, title)
    _write_summary_line(f, "=" * 80)

    scalar_fields = [
        ("Total Execution Time (s)", lambda record: record.get("elapsed")),
        ("Mean Exp Reward", lambda record: record["metrics"].get("mean_exp_reward")),
        ("Unique Sample Exp Reward Sum", lambda record: get_unique_sample_exp_reward_sum(record["metrics"])),
        ("Diversity", lambda record: record["metrics"].get("diversity")),
        ("Uniqueness (similarity > 0.85)", lambda record: record["metrics"].get("uniqueness")),
        ("Unique Valid Count (r(x_0) > -0.1, similarity > 0.85)", lambda record: record["metrics"].get("unique_valid_count")),
    ]

    for name, getter in scalar_fields:
        stats = _summary_stats([getter(record) for record in records])
        if stats is not None:
            _write_summary_line(f, _format_summary_stats(name, stats))


LATEX_POLICY_ROWS = [
    ("allstep", "Full", False),
    ("uniform", "Uniform", False),
    ("top_delta_v", "Top dV", False),
    ("top_v", "Top V", False),
    ("adaptive_dp", "VISTA", False),
    ("interval1", "Interval 1 (First)", False),
    ("interval2", "Interval 2", False),
    ("interval3", "Interval 3", False),
    ("interval4", "Interval 4", False),
    ("interval5", "Interval 5 (Last)", False),
]


def _policy_summary_stats(records: list[dict[str, Any]], policy_name: str):
    policy_records = [record for record in records if record.get("policy") == policy_name]
    if not policy_records:
        return None
    return {
        "time": _summary_stats([record.get("elapsed") for record in policy_records]),
        "unique_valid": _summary_stats([record["metrics"].get("unique_valid_count") for record in policy_records]),
        "mean_exp_reward": _summary_stats([record["metrics"].get("mean_exp_reward") for record in policy_records]),
    }


def _latex_mean_pm(stats: dict[str, Any] | None, digits: int, pm: str = r"$\pm$"):
    if stats is None:
        return "{--}"
    return f"{stats['mean']:.{digits}f} {pm} {stats['std']:.{digits}f}"


def _latex_number(value: float | None, bold: bool = False):
    if value is None:
        return "{--}"
    value = float(value)
    abs_value = abs(value)
    if abs_value > 0 and (abs_value < 0.1 or abs_value >= 100):
        exponent = int(np.floor(np.log10(abs_value)))
        mantissa = value / (10 ** exponent)
        body = f"{mantissa:.3f}\\times10^{{{exponent}}}"
    else:
        body = f"{value:.3f}"
    if bold:
        return f"$\\mathbf{{{body}}}$"
    return f"${body}$"


def _policy_utd(policy_name: str):
    if policy_name == "allstep":
        return None
    value_sum_names = {"adaptive_dp": "result_sum", "uniform": "result_sum_unif", "top_delta_v": "result_sum_top_delta_v", "top_v": "result_sum_top_v", **{f"interval{k}": f"result_sum_interval{k}" for k in range(1, 6)}}
    value_sum_name = value_sum_names.get(policy_name)
    if value_sum_name is None or value_sum_name not in globals():
        return None
    value_sum = globals()[value_sum_name]
    return float((num_timesteps - N_GUIDANCE_STEPS - value_sum) * 2 / num_timesteps)


def write_latex_tstep_table(f, records: list[dict[str, Any]]):
    stats_by_policy = {policy_name: _policy_summary_stats(records, policy_name) for policy_name, _, _ in LATEX_POLICY_ROWS}
    budget_label = rf"$T'={N_GUIDANCE_STEPS}$"
    first_block_rows = [(policy_name, schedule_label, is_method) for policy_name, schedule_label, is_method in LATEX_POLICY_ROWS if not is_method and stats_by_policy.get(policy_name) is not None]
    method_rows = [(policy_name, schedule_label, is_method) for policy_name, schedule_label, is_method in LATEX_POLICY_ROWS if is_method and stats_by_policy.get(policy_name) is not None]
    if not first_block_rows and not method_rows:
        return
    row_count = len(first_block_rows) + len(method_rows)
    _write_summary_line(f, "")
    _write_summary_line(f, "=" * 80)
    _write_summary_line(f, "LaTeX table rows")
    _write_summary_line(f, "=" * 80)
    _write_summary_line(f, r"\textbf{Budget}")
    _write_summary_line(f, r"  & \textbf{Schedule}")
    _write_summary_line(f, r"  & {\textbf{Time [s]} $\downarrow$}")
    _write_summary_line(f, r"  & {\textbf{Unique valid} $\uparrow$}")
    _write_summary_line(f, r"  & {\textbf{Exp reward mean} $\uparrow$}")
    _write_summary_line(f, r"  & \textbf{UTD} $\downarrow$ \\")
    _write_summary_line(f, r"\midrule")
    _write_summary_line(f, "")
    for row_index, (policy_name, schedule_label, _) in enumerate(first_block_rows):
        policy_stats = stats_by_policy.get(policy_name)
        prefix = rf"\multirow{{{row_count}}}{{*}}{{{budget_label}}}" if row_index == 0 else ""
        utd = _policy_utd(policy_name)
        line = f"{prefix}  & {schedule_label:<7} & {_latex_mean_pm(policy_stats['time'], 2, pm='+-')} & {_latex_mean_pm(policy_stats['unique_valid'], 2, pm='+-')} & {_latex_mean_pm(policy_stats['mean_exp_reward'], 4, pm='+-')} & {_latex_number(utd)} \\\\"
        _write_summary_line(f, line)
    if first_block_rows and method_rows:
        _write_summary_line(f, r"\cmidrule(lr){2-6}")
    for policy_name, schedule_label, _ in method_rows:
        policy_stats = stats_by_policy.get(policy_name)
        utd = _policy_utd(policy_name)
        line = (
            f"  & {schedule_label}"
            f" & \\multicolumn{{1}}{{c}}{{\\textbf{{{_latex_mean_pm(policy_stats['time'], 2)}}}}}"
            f" & \\multicolumn{{1}}{{c}}{{\\textbf{{{_latex_mean_pm(policy_stats['unique_valid'], 2)}}}}}"
            f" & \\multicolumn{{1}}{{c}}{{\\textbf{{{_latex_mean_pm(policy_stats['mean_exp_reward'], 4)}}}}}"
            f" & {_latex_number(utd, bold=True)} \\\\"
        )
        _write_summary_line(f, line)


## Warmup, DP, And Benchmark


In [ ]:
def run_allstep_warmup(seed_range=WARMUP_SEEDS):
    cell_start_wall = time.time()
    output_file = result_path("allstep_warmup", shared=True)
    existing_output_file = find_existing_result_path("allstep_warmup", shared=True)
    if existing_output_file is not None:
        try:
            mean_exp_rewards_avg, mean_exp_rewards_std = load_warmup_mean_exp_rewards(existing_output_file)
        except ValueError as exc:
            print(f"Ignoring incomplete allstep warmup cache: {existing_output_file} ({exc})")
        else:
            print(f"Skipping allstep warmup; found existing result file: {existing_output_file}")
            return existing_output_file, mean_exp_rewards_avg, mean_exp_rewards_std
    all_mean_exp_rewards = []
    summary_records = []
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write(f"Sequential Monte Carlo {PROPOSAL_MODE.title()} Reduced 실험 결과\n")
        f.write(f"Model type: {MODEL_TYPE}\n")
        f.write(METRIC_SEMANTICS_LINE + "\n")
        f.write(f"Shared allstep warmup across N_GUIDANCE_STEPS: {list(N_GUIDANCE_STEPS_SWEEP)}\n")
        f.write("=" * 80 + "\n\n")
        for seed in seed_range:
            seed_info = f"Seed: {seed}\n"
            print(seed_info, end="")
            f.write(seed_info)
            _, metrics, elapsed = run_smc_once(seed, set(range(0, num_timesteps)))
            time_info = f"Total Execution Time: {elapsed:.4f} seconds\n"
            print(time_info, end="")
            f.write(time_info)
            write_metrics(f, metrics)
            summary_records.append({"metrics": metrics, "elapsed": elapsed})
            mean_exp_rewards = metrics.get("mean_exp_rewards")
            if mean_exp_rewards is not None:
                all_mean_exp_rewards.append(np.array(mean_exp_rewards))
        mean_exp_rewards_avg, mean_exp_rewards_std = summarize_mean_exp_rewards(all_mean_exp_rewards)
        if mean_exp_rewards_avg is None or len(mean_exp_rewards_avg) == 0:
            raise RuntimeError("Allstep warmup did not produce per-timestep mean rewards; cannot solve tstep policies.")
        f.write("\n" + "=" * 80 + "\n")
        f.write("Timestep별 Mean Expected Reward 평균 및 표준편차 (seed별)\n")
        f.write("=" * 80 + "\n")
        print("\n" + "=" * 80)
        print("Timestep별 Mean Expected Reward 평균 및 표준편차 (seed별)")
        print("=" * 80)
        for t in range(len(mean_exp_rewards_avg)):
            timestep_info = f"Timestep {t}: mean={mean_exp_rewards_avg[t]:.4f} std={mean_exp_rewards_std[t]:.4f}\n"
            print(timestep_info, end="")
            f.write(timestep_info)
        write_metric_summary(f, "Summary averages | allstep warmup", summary_records)
        cell_end_wall = time.time()
        cell_total_s = cell_end_wall - cell_start_wall
        f.write(f"[Cell Timing] warmup(allstep) total_seconds: {cell_total_s:.4f}\n")
        f.write("=" * 80 + "\n")
    print(f"\n결과가 '{output_file}' 파일에 저장되었습니다.")
    return output_file, mean_exp_rewards_avg, mean_exp_rewards_std


def solve_adaptive_steering_dp(arr, T_prime_count):
    T_max = len(arr) - 1
    dp = np.full((T_max, T_prime_count + 1), -1.0)
    parent = np.zeros((T_max, T_prime_count + 1), dtype=np.int64)
    V_T = arr[T_max]
    for s in range(T_max):
        dp[s][1] = (T_max - 1 - s) * V_T
    for i in range(2, T_prime_count + 1):
        for s in range(T_max - i + 1):
            s_prime_range = np.arange(s + 1, T_max - i + 2)
            candidate_vals = dp[s_prime_range, i - 1] + (s_prime_range - s - 1) * np.array([arr[sp] for sp in s_prime_range])
            best_idx = np.argmax(candidate_vals)
            dp[s][i] = candidate_vals[best_idx]
            parent[s][i] = s_prime_range[best_idx]
    i = T_prime_count
    s_range = np.arange(0, T_max - i + 1)
    scores = dp[s_range, i] + s_range * np.asarray(arr)[s_range]
    t_start = int(s_range[np.argmax(scores)])
    max_total_sum = float(dp[t_start, T_prime_count] + t_start * np.asarray(arr)[t_start])
    indices = []
    curr_s = t_start
    for i in range(T_prime_count, 0, -1):
        indices.append(int(curr_s))
        if i > 1:
            curr_s = parent[curr_s][i]
    return max_total_sum, sorted(indices)


def calculate_guidance_sum(indices, arr):
    T_max = len(arr) - 1
    sorted_indices = sorted(list(indices))
    total_val_sum = 0
    t_start = sorted_indices[0]
    total_val_sum += t_start * arr[t_start]
    for i in range(len(sorted_indices) - 1):
        s = sorted_indices[i]
        s_next = sorted_indices[i + 1]
        count = s_next - s - 1
        if count > 0:
            total_val_sum += count * arr[s_next]
    t_last = sorted_indices[-1]
    count_last = T_max - 1 - t_last
    if count_last > 0:
        total_val_sum += count_last * arr[T_max]
    return total_val_sum


def top_k_schedule_by_score(scores, count):
    scores = np.asarray(scores, dtype=float)
    if count < 1 or count > len(scores):
        raise ValueError(f"Cannot choose {count} timesteps from {len(scores)} scores")
    ranked = sorted(range(len(scores)), key=lambda index: (-scores[index], index))
    return {int(index) for index in ranked[:count]}


def solve_top_delta_v_schedule(arr, T_prime_count):
    values = np.asarray(arr, dtype=float)
    delta_values = values[:-1] - values[1:]
    return top_k_schedule_by_score(delta_values, T_prime_count)


def solve_top_v_schedule(arr, T_prime_count):
    values = np.asarray(arr, dtype=float)
    return top_k_schedule_by_score(values[:-1], T_prime_count)


def interval_schedule(num_timesteps: int, t_prime_count: int, interval_index: int):
    if t_prime_count < 1 or t_prime_count > num_timesteps:
        raise ValueError(f"Cannot choose {t_prime_count} timesteps from {num_timesteps} timesteps")
    if interval_index not in range(1, 6):
        raise ValueError("interval_index must be between 1 and 5")
    start = round((num_timesteps - t_prime_count) / 4 * (5 - interval_index))
    return set(range(start, start + t_prime_count))


def solve_and_register_tstep_policies(mean_exp_rewards_avg, warmup_output_file):
    global result_sum, indices, T_prime, T_unif, T_top_delta_v, T_top_v
    global result_sum_unif, result_sum_top_delta_v, result_sum_top_v
    global T_interval1, T_interval2, T_interval3, T_interval4, T_interval5
    global result_sum_interval1, result_sum_interval2, result_sum_interval3, result_sum_interval4, result_sum_interval5
    if mean_exp_rewards_avg is None:
        raise RuntimeError("Run the all-step warmup before solving the DP.")
    cell_start_wall = time.time()
    N_ = N_GUIDANCE_STEPS
    arr = list(mean_exp_rewards_avg)
    result_sum, indices = solve_adaptive_steering_dp(arr, N_)
    T_prime = {int(i) for i in indices}
    T_unif = {round(k * (num_timesteps - 1) / (N_ - 1)) for k in range(N_)} if N_ > 1 else {0}
    T_top_delta_v = solve_top_delta_v_schedule(arr, N_)
    T_top_v = solve_top_v_schedule(arr, N_)
    interval_schedules = {k: interval_schedule(num_timesteps, N_, k) for k in range(1, 6)}
    T_interval1, T_interval2, T_interval3, T_interval4, T_interval5 = (interval_schedules[k] for k in range(1, 6))

    result_sum_unif = calculate_guidance_sum(T_unif, arr)
    result_sum_top_delta_v = calculate_guidance_sum(T_top_delta_v, arr)
    result_sum_top_v = calculate_guidance_sum(T_top_v, arr)
    interval_value_sums = {k: calculate_guidance_sum(interval_schedules[k], arr) for k in range(1, 6)}
    result_sum_interval1, result_sum_interval2, result_sum_interval3, result_sum_interval4, result_sum_interval5 = (interval_value_sums[k] for k in range(1, 6))

    print(f"VISTA schedule: {sorted(T_prime)} | value_sum={result_sum}")
    print(f"Uniform schedule: {sorted(T_unif)} | value_sum={result_sum_unif}")
    print(f"Top dV schedule: {sorted(T_top_delta_v)} | value_sum={result_sum_top_delta_v}")
    print(f"Top V schedule: {sorted(T_top_v)} | value_sum={result_sum_top_v}")
    for k in range(1, 6):
        print(f"Interval {k} schedule: {sorted(interval_schedules[k])} | value_sum={interval_value_sums[k]}")

    cell_total_s = time.time() - cell_start_wall
    with open(warmup_output_file, "a", encoding="utf-8") as f:
        f.write("\n" + "=" * 80 + "\n")
        f.write("Schedule construction\n")
        f.write("=" * 80 + "\n")
        f.write(f"N_GUIDANCE_STEPS: {N_}\n")
        f.write(f"[Cell Timing] schedule total_seconds: {cell_total_s:.4f}\n")
        f.write(f"VISTA value sum: {result_sum}\n")
        f.write(f"VISTA schedule (sorted): {sorted(T_prime)}\n")
        f.write(f"Uniform value sum: {result_sum_unif}\n")
        f.write(f"Uniform schedule (sorted): {sorted(T_unif)}\n")
        f.write(f"Top dV value sum: {result_sum_top_delta_v}\n")
        f.write(f"Top dV schedule (sorted): {sorted(T_top_delta_v)}\n")
        f.write(f"Top V value sum: {result_sum_top_v}\n")
        f.write(f"Top V schedule (sorted): {sorted(T_top_v)}\n")
        for k in range(1, 6):
            f.write(f"Interval {k} value sum: {interval_value_sums[k]}\n")
            f.write(f"Interval {k} schedule (sorted): {sorted(interval_schedules[k])}\n")

    return {
        "allstep": set(range(num_timesteps)),
        "uniform": T_unif,
        "top_delta_v": T_top_delta_v,
        "top_v": T_top_v,
        "adaptive_dp": T_prime,
        **{f"interval{k}": interval_schedules[k] for k in range(1, 6)},
    }


def _clone_policy_records(records: list[dict[str, Any]], cached_from: str):
    cloned = []
    for record in records:
        new_record = dict(record)
        new_record["cached"] = True
        new_record["cached_from"] = cached_from
        cloned.append(new_record)
    return cloned


def find_cached_allstep_records():
    existing_output_file = find_existing_result_path("allstep")
    if existing_output_file is None:
        return None, None
    records = [record for record in parse_result_records(existing_output_file) if record.get("policy") == "allstep"]
    return records, str(existing_output_file)


def run_allstep_benchmark(seed_range=EVAL_SEEDS):
    output_file = result_path("allstep")
    existing_output_file = find_existing_result_path("allstep")
    if existing_output_file is not None:
        summary_records = parse_result_records(existing_output_file)
        print(f"Skipping allstep benchmark; found existing result file: {existing_output_file}")
        return existing_output_file, summary_records
    summary_records = []
    allstep_tstep = set(range(0, num_timesteps))
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write(f"Sequential Monte Carlo {PROPOSAL_MODE.title()} Allstep 실험 결과\n")
        f.write(f"Model type: {MODEL_TYPE}\n")
        f.write(METRIC_SEMANTICS_LINE + "\n")
        f.write("Policy: allstep\n")
        f.write("=" * 80 + "\n\n")
        for seed in seed_range:
            header = f"Seed: {seed} | Policy: allstep | tstep_count: {len(allstep_tstep)}\n"
            print(header, end="")
            f.write(header)
            _, metrics, elapsed = run_smc_once(seed, allstep_tstep)
            time_info = f"Total Execution Time: {elapsed:.4f} seconds\n"
            print(time_info, end="")
            f.write(time_info)
            write_metrics(f, metrics)
            summary_records.append({"policy": "allstep", "tstep_count": len(allstep_tstep), "metrics": metrics, "elapsed": elapsed})
            f.write("\n")
        write_metric_summary(f, "Summary averages | Policy: allstep", summary_records)
        write_metric_summary(f, "Summary averages | Overall", summary_records)
        f.write("=" * 80 + "\n")
    print(f"\n결과가 '{output_file}' 파일에 저장되었습니다.")
    return output_file, summary_records


def run_tstep_benchmark(tstep_policies: dict[str, set[int]] | None = None, seed_range=EVAL_SEEDS, cached_policy_records: dict[str, list[dict[str, Any]]] | None = None):
    output_file = result_path("tstep")
    compatible_output_file = find_existing_result_path("tstep")
    if compatible_output_file is not None:
        summary_records = parse_result_records(compatible_output_file)
        print(f"Skipping tstep benchmark; found complete result file: {compatible_output_file}")
        return compatible_output_file, summary_records
    if tstep_policies is None:
        tstep_policies = {"uniform": T_unif, "top_delta_v": T_top_delta_v, "top_v": T_top_v, "adaptive_dp": T_prime, "interval1": T_interval1, "interval2": T_interval2, "interval3": T_interval3, "interval4": T_interval4, "interval5": T_interval5}
    else:
        tstep_policies = {policy_name: tstep for policy_name, tstep in tstep_policies.items() if policy_name != "allstep"}
    seed_range = [int(seed) for seed in seed_range]
    cached_records = []
    if output_file.exists() and has_compatible_metric_semantics(output_file):
        cached_records.extend(parse_result_records(output_file))
    temporary_file = output_file.with_suffix(output_file.suffix + ".incremental.tmp")
    if temporary_file.exists() and has_compatible_metric_semantics(temporary_file):
        temporary_records = parse_result_records(temporary_file)
        cached_records.extend(temporary_records)
        print(f"Resuming {len(temporary_records)} records from interrupted incremental output: {temporary_file}")
    for policy_name, records in (cached_policy_records or {}).items():
        if policy_name != "allstep":
            cached_records.extend(records)
    cached_by_key = {}
    policy_aliases = {"firstk": "interval1", "lastk": "interval5"}
    for record in cached_records:
        policy_name = policy_aliases.get(record.get("policy"), record.get("policy"))
        key = (record.get("seed"), policy_name)
        if key[0] in seed_range and key[1] in tstep_policies:
            cached_record = dict(record)
            cached_record["policy"] = policy_name
            cached_by_key[key] = cached_record
    expected_keys = {(seed, policy_name) for seed in seed_range for policy_name in tstep_policies}
    missing_keys = expected_keys - set(cached_by_key)
    print(f"Tstep incremental cache: reused={len(cached_by_key)}, missing={len(missing_keys)}")
    backup_file = output_file.with_suffix(output_file.suffix + ".before_incremental")
    if output_file.exists() and not backup_file.exists():
        backup_file.write_bytes(output_file.read_bytes())
        print(f"Created result backup: {backup_file}")
    summary_records = []
    with open(temporary_file, "w", encoding="utf-8") as f:
        f.write("=" * 80 + "\n")
        f.write(f"Sequential Monte Carlo {PROPOSAL_MODE.title()} Reduced 실험 결과\n")
        f.write(f"Model type: {MODEL_TYPE}\n")
        f.write(METRIC_SEMANTICS_LINE + "\n")
        f.write(f"N_GUIDANCE_STEPS: {N_GUIDANCE_STEPS}\n")
        f.write("=" * 80 + "\n\n")
        for seed in seed_range:
            for policy_name, tstep in tstep_policies.items():
                header = f"Seed: {seed} | Policy: {policy_name} | tstep_count: {len(tstep)}\n"
                f.write(header)
                cached_record = cached_by_key.get((seed, policy_name))
                if cached_record is not None:
                    metrics = cached_record["metrics"]
                    elapsed = cached_record["elapsed"]
                    print(f"Reusing cached seed={seed} policy={policy_name}")
                else:
                    print(header, end="")
                    _, metrics, elapsed = run_smc_once(seed, tstep)
                time_info = f"Total Execution Time: {elapsed:.4f} seconds\n"
                if cached_record is None:
                    print(time_info, end="")
                f.write(time_info)
                write_metrics(f, metrics)
                summary_records.append({"seed": seed, "policy": policy_name, "tstep_count": len(tstep), "metrics": metrics, "elapsed": elapsed})
                f.write("\n")
        policy_names = []
        for policy_name, _, _ in LATEX_POLICY_ROWS:
            if any(record["policy"] == policy_name for record in summary_records):
                policy_names.append(policy_name)
        for policy_name in policy_names:
            policy_records = [record for record in summary_records if record["policy"] == policy_name]
            write_metric_summary(f, f"Summary averages | Policy: {policy_name}", policy_records)
        write_metric_summary(f, "Summary averages | Overall", summary_records)
        write_latex_tstep_table(f, summary_records)
        f.write("=" * 80 + "\n")
    temporary_file.replace(output_file)
    print(f"\n결과가 '{output_file}' 파일에 저장되었습니다.")
    return output_file, summary_records


## Run Sweep


In [ ]:
def run_single_experiment(model_type: str, proposal_mode: str, n_guidance_steps: int):
    context = set_experiment_context(model_type, proposal_mode, n_guidance_steps)
    set_seed(SEED)
    warmup_output_file, mean_exp_rewards_avg, mean_exp_rewards_std = run_allstep_warmup()
    allstep_output_file, allstep_records = run_allstep_benchmark()
    tstep_policies = None
    if find_existing_result_path("tstep") is None:
        tstep_policies = solve_and_register_tstep_policies(mean_exp_rewards_avg, warmup_output_file)
    tstep_output_file, summary_records = run_tstep_benchmark(tstep_policies=tstep_policies)
    return {**context, "warmup_output_file": warmup_output_file, "allstep_output_file": allstep_output_file, "tstep_output_file": tstep_output_file, "allstep_records": allstep_records, "summary_records": summary_records, "mean_exp_rewards_avg": mean_exp_rewards_avg, "mean_exp_rewards_std": mean_exp_rewards_std}


def run_experiment_sweep(model_types=MODEL_TYPE_SWEEP, proposal_modes=PROPOSAL_MODE_SWEEP, n_guidance_steps_list=N_GUIDANCE_STEPS_SWEEP):
    total = len(model_types) * len(proposal_modes) * len(n_guidance_steps_list)
    records = []
    sweep_start = time.time()
    print("=" * 80)
    print("Starting experiment sweep")
    print(f"Total configs: {total}")
    print("Allstep warmup and allstep policy are run once per MODEL_TYPE / PROPOSAL_MODE.")
    print("=" * 80)
    config_index = 0
    for model_type in model_types:
        for proposal_mode in proposal_modes:
            print("\n" + "@" * 80)
            print(f"Preparing shared allstep context: MODEL_TYPE={model_type}, PROPOSAL_MODE={proposal_mode}")
            print("@" * 80)
            first_n_guidance_steps = n_guidance_steps_list[0]
            context = set_experiment_context(model_type, proposal_mode, first_n_guidance_steps)
            set_seed(SEED)
            warmup_output_file, mean_exp_rewards_avg, mean_exp_rewards_std = run_allstep_warmup()
            allstep_output_file, allstep_records = run_allstep_benchmark()
            for n_guidance_steps in n_guidance_steps_list:
                config_index += 1
                set_guidance_steps(n_guidance_steps)
                print("\n" + "#" * 80)
                print(f"Sweep config {config_index}/{total}: MODEL_TYPE={model_type}, PROPOSAL_MODE={proposal_mode}, N_GUIDANCE_STEPS={n_guidance_steps}")
                print("#" * 80)
                if find_existing_result_path("tstep") is None:
                    tstep_policies = solve_and_register_tstep_policies(mean_exp_rewards_avg, warmup_output_file)
                    tstep_output_file, summary_records = run_tstep_benchmark(tstep_policies=tstep_policies)
                else:
                    tstep_output_file, summary_records = run_tstep_benchmark()
                records.append({**context, "sweep_index": config_index, "n_guidance_steps": n_guidance_steps, "warmup_output_file": warmup_output_file, "allstep_output_file": allstep_output_file, "tstep_output_file": tstep_output_file, "mean_exp_rewards_avg": mean_exp_rewards_avg, "mean_exp_rewards_std": mean_exp_rewards_std})
            clear_active_model()
    elapsed = time.time() - sweep_start
    print("\n" + "=" * 80)
    print(f"Experiment sweep completed in {elapsed:.4f} seconds")
    print("=" * 80)
    return records


experiment_sweep_records = run_experiment_sweep()
